# Setup

In [1]:
%load_ext autoreload
%autoreload 2

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")

from finintel.agent.graph import AgentPipeline

agent = AgentPipeline()
print(f"Model: {agent.model}")
print(f"Per-query K: {agent.per_query_k}")
print(f"Max context chunks: {agent.max_context_chunks}")

c:\Users\User\Desktop\portfolio_projects\ML_projects\finintel\.venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
16:01:23 [INFO] Loading embedding model: BAAI/bge-base-en-v1.5
16:01:23 [INFO] No device provided, using cpu
16:01:23 [INFO] Loading SentenceTransformer model from BAAI/bge-base-en-v1.5.
16:01:24 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
16:01:24 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en-v1.5/a5beb1e3e68b9ab74eb54cfd186867f64f240e1a/README.md "HTTP/1.1 200 OK"
16:01:24 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

16:01:26 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
16:01:26 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
16:01:26 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
16:01:27 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
16:01:27 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
16:01:27 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en-v1.5/a5beb1e3e68b9ab74eb54cfd186867f64f240e1a/tokenizer_config.json "HTTP/1.1 200 OK"
16:01:28 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/ma

Model: llama-3.1-8b-instant
Per-query K: 2
Max context chunks: 4


# The multi-company question that broke baseline

In [2]:
result = agent.answer(
    "How do Google and Microsoft each frame competitive risks in their cloud businesses?"
)

print("=" * 70)
print("PLANNER OUTPUT")
print("=" * 70)
for sq in result.sub_queries:
    print(f"  • [{sq.get('ticker') or 'any'}/{sq.get('section') or 'any'}] {sq['question']}")

print("\n" + "=" * 70)
print("ANSWER")
print("=" * 70)
print(result.answer)

print("\n" + "=" * 70)
print(f"SOURCES ({len(result.sources)})")
print("=" * 70)
from collections import Counter
print("By ticker:", dict(Counter(s["ticker"] for s in result.sources)))
print("By section:", dict(Counter(s["section"] for s in result.sources)))
for s in result.sources:
    print(f"  - {s['ticker']:6} {s['section']:14} {s['chunk_id']}")

print(f"\nTokens: {result.input_tokens:,} in / {result.output_tokens:,} out")

16:01:39 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
16:01:39 [INFO] Planner produced 2 sub-queries
16:01:40 [INFO] HTTP Request: POST http://localhost:6333/collections/finintel_chunks/points/query "HTTP/1.1 200 OK"
16:01:40 [INFO] HTTP Request: POST http://localhost:6333/collections/finintel_chunks/points/query "HTTP/1.1 200 OK"
16:01:40 [INFO] Retriever accumulated 4 unique chunks
16:01:41 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


PLANNER OUTPUT
  • [GOOGL/risk_factors] Google's cloud business competitive risks framing
  • [MSFT/risk_factors] Microsoft's cloud business competitive risks framing

ANSWER
To address the question, I will frame the competitive risks in Google's and Microsoft's cloud businesses based solely on the provided filing excerpts.

**Google Cloud:**

Google frames competitive risks in its cloud business through the following:

- High competition from rapidly evolving cloud-based services offered by competitors.
- Competitive pricing and delivery models.
- Difficulty achieving sufficient scale and profitability, despite devoting significant resources to develop and deploy cloud services.
- Challenges in complying with evolving regulations and laws, including government audits and cost reviews, which may expose Google to legal, financial, and reputational risks.
- Intense competition from large, experienced, and well-funded competitors in emerging technology areas, such as health, life sciences

# Compare against the baseline single-shot

In [3]:
from finintel.agent.generator import RAGPipeline

baseline = RAGPipeline()
baseline_result = baseline.answer(
    "How do Google and Microsoft each frame competitive risks in their cloud businesses?"
)

print("BASELINE source distribution:")
print(dict(Counter(s["ticker"] for s in baseline_result.sources)))

print("\nAGENT source distribution:")
print(dict(Counter(s["ticker"] for s in result.sources)))

16:01:57 [INFO] Loading embedding model: BAAI/bge-base-en-v1.5
16:01:57 [INFO] No device provided, using cpu
16:01:57 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
16:01:57 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en-v1.5/a5beb1e3e68b9ab74eb54cfd186867f64f240e1a/modules.json "HTTP/1.1 200 OK"
16:01:57 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
16:01:57 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en-v1.5/a5beb1e3e68b9ab74eb54cfd186867f64f240e1a/config_sentence_transformers.json "HTTP/1.1 200 OK"
16:01:57 [INFO] Loading SentenceTransformer model from BAAI/bge-base-en-v1.5.
16:01:58 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

16:02:00 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
16:02:00 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
16:02:00 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
16:02:01 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
16:02:01 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
16:02:01 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en-v1.5/a5beb1e3e68b9ab74eb54cfd186867f64f240e1a/tokenizer_config.json "HTTP/1.1 200 OK"
16:02:01 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/ma

BASELINE source distribution:
{'GOOGL': 3, 'MSFT': 1}

AGENT source distribution:
{'GOOGL': 2, 'MSFT': 2}
